In [1]:
import torch
import torch.nn as nn
device = torch.device('cuda:0')

In [6]:
a = [1,2,3]

In [7]:
a.pop(0)

1

In [8]:
a.append(4)

In [9]:
a

[2, 3, 4]

In [11]:
class MemoryAttention(nn.Module):
    def __init__(self, d_model, n_head, hid_width, memory_len=1, current_len=1):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.hid_width = hid_width
        self.memory_len = memory_len
        self.current_len = current_len
        self.d_head = d_model // n_head
        self.q_linear = nn.Linear(d_model, d_model, bias=False)
        self.k_linear = nn.Linear(d_model, d_model, bias=False)
        self.v_linear = nn.Linear(d_model, d_model, bias=False)
        self.att_linear = nn.Linear(d_model, d_model, bias=False)
        self.q_weight = nn.Parameter(torch.stack([torch.ones(n_head,self.d_head//2+1),torch.zeros(n_head,self.d_head//2+1)],dim=-1))
        self.k_weight = nn.Parameter(torch.stack([torch.ones(n_head,self.d_head//2+1),torch.zeros(n_head,self.d_head//2+1)],dim=-1))
        self.register_buffer('theta_head', 10000**(-torch.arange(self.d_head//2+1)/(self.d_head//2+1)))
        self.nonlinear = nn.Sequential(
            nn.Linear(self.d_model,hid_width),
            nn.GELU(),
            nn.Linear(hid_width,self.d_model)
        )

    def forward(self, token_seq):
        q, k, v = self._get_qkv(token_seq)

        token_att = self._multi_head_attention(q, k, v)

        return self.att_linear(token_att)
    
    def _multi_head_attention(self, q, k, v):
        mem_k = []
        cur_k = []
        mem_v = []
        cur_v = []
        ot = []
        for t in range(q.shape[1]):
            cur_k.append(k[:,t])
            cur_v.append(v[:,t])
            kt = torch.stack(mem_k + cur_k, dim=1)
            vt = torch.stack(mem_v + cur_v, dim=1)
            st = torch.einsum('b n d, b t n d -> b t n', q[:,t], kt) / self.d_head
            ot.append(torch.einsum('b t n, b t n d -> b n d', torch.softmax(st, dim=1), vt))

            len_mem = len(mem_k)
            len_cur = len(cur_k)
            mem_k.append(torch.einsum('b t n, b t n d -> b n d', torch.softmax(st[:,:(len_mem+1)], dim=1), kt[:,:(len_mem+1)]))
            mem_v.append(torch.einsum('b t n, b t n d -> b n d', torch.softmax(st[:,:(len_mem+1)], dim=1), vt[:,:(len_mem+1)]))
            cur_k.pop(0)
            cur_v.pop(0)
            if len_mem + len_cur > self.memory_len + self.current_len:
                mem_k.pop(0)
                mem_v.pop(0)
        
        return self.att_linear(torch.stack(ot, dim=1).flatten(-2,-1))
            

    def _get_qkv(self, token_seq):
        with torch.no_grad():
            t = torch.arange(token_seq.shape[1],device=token_seq.device)
            pe_weight = torch.exp(1j*t[:,None,None]*self.theta_head[None,None,:])
        q_weight = torch.view_as_complex(self.q_weight)
        k_weight = torch.view_as_complex(self.k_weight)
        q = self.q_linear(token_seq).view(*token_seq.shape[:2],self.n_head,self.d_head)
        k = self.k_linear(token_seq).view(*token_seq.shape[:2],self.n_head,self.d_head)
        
        q = torch.fft.irfft(torch.fft.rfft(q) * pe_weight * q_weight)
        k = torch.fft.irfft(torch.fft.rfft(k) * pe_weight * k_weight)

        v = self.v_linear(token_seq).view(*token_seq.shape[:2],self.n_head,self.d_head)
        return q, k, v
    
token_seq = torch.rand((4,10,512), device=device)
layer = MemoryAttention(512, 8, 2048, memory_len=5, current_len=5).to(device)
out = layer(token_seq)

In [12]:
out.shape

torch.Size([4, 10, 512])

In [16]:
import torch
import cying.nn as cynn
device = torch.device('cuda:0')

from torchinfo import summary
num_class = 10**4
opt_lang_model = cynn.MemoryLangModel(
    num_class,
    512,
    4,
    2048,
    6
).to(device)
summary(opt_lang_model)

Layer (type:depth-idx)                   Param #
MemoryLangModel                          --
├─Sequential: 1-1                        --
│    └─Embedding: 2-1                    5,120,000
│    └─RMSNorm: 2-2                      512
├─Sequential: 1-2                        --
│    └─MemoryAttLayer: 2-3               520
│    │    └─Linear: 3-1                  262,144
│    │    └─Linear: 3-2                  262,144
│    │    └─Linear: 3-3                  262,144
│    │    └─Linear: 3-4                  262,144
│    │    └─Sequential: 3-5              2,099,712
│    │    └─RMSNorm: 3-6                 512
│    │    └─RMSNorm: 3-7                 512
│    └─MemoryAttLayer: 2-4               520
│    │    └─Linear: 3-8                  262,144
│    │    └─Linear: 3-9                  262,144
│    │    └─Linear: 3-10                 262,144
│    │    └─Linear: 3-11                 262,144
│    │    └─Sequential: 3-12             2,099,712
│    │    └─RMSNorm: 3-13                512
│   

In [ ]:
import json
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader

scale = 64
max_seq_len = 512
batch_size = 1

class Trans2019(Dataset):
    def __init__(self, filepath, tokenizer, seq_len=64):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.data_english = []
        self.data_chinese = []
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                datai = json.loads(line)
                self.data_english.append(datai['english'])
                self.data_chinese.append(datai['chinese'])
    
    def __getitem__(self, index):
        text_english = self.data_english[index]
        text_chinese = self.data_chinese[index]
        seq_len = len(text_english) + len(text_chinese)
        cut_rand = torch.rand(1)
        while seq_len/self.seq_len < cut_rand:
            index_rand = torch.randint(0,self.__len__(),(1,))[0]
            text_english += self.data_english[index_rand]
            text_chinese += self.data_chinese[index_rand]
            seq_len = len(text_english) + len(text_chinese)
        if torch.randn(1) < 0:
            tokens_english = [0] + self.tokenizer.encode(text_english).ids + [1]
            tokens_chinese = [2] + self.tokenizer.encode(text_chinese).ids + [3]
            token_seq = tokens_english + tokens_chinese[:-1]
            tgt_seq = [4]*len(tokens_english) + tokens_chinese[1:]
        else:
            tokens_english = [2] + self.tokenizer.encode(text_english).ids + [3]
            tokens_chinese = [0] + self.tokenizer.encode(text_chinese).ids + [1]
            token_seq = tokens_chinese + tokens_english[:-1]
            tgt_seq = [4]*len(tokens_chinese) + tokens_english[1:]
        if len(token_seq) >= self.seq_len:
            token_seq = token_seq[:self.seq_len]
            tgt_seq = tgt_seq[:self.seq_len]
        else:
            token_seq = token_seq + [4]*(self.seq_len - len(token_seq))
            tgt_seq = tgt_seq + [4]*(self.seq_len - len(tgt_seq))
        input_ids = torch.tensor(token_seq, dtype=torch.long)
        target_ids = torch.tensor(tgt_seq, dtype=torch.long)
        return input_ids, target_ids

    def __len__(self):
        return len(self.data_english) // batch_size * batch_size

tokenizer = Tokenizer.from_file("./cying/datasets/tokenizer-wiki.json")
trans_dataset = Trans2019(
    filepath=f'./cying/datasets/translation2019zh/translation2019zh_train.json',
    tokenizer=tokenizer,
    seq_len=max_seq_len
)
train_loader = DataLoader(
    dataset=trans_dataset,
    batch_size=batch_size,
    shuffle=True
)
valid_dataset = Trans2019(
    filepath=f'./cying/datasets/translation2019zh/translation2019zh_valid.json',
    tokenizer=tokenizer,
    seq_len=max_seq_len
)
valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=batch_size
)

from tqdm import tqdm
epoch = 10**2
lr = 1e-3
optim = torch.optim.Adam(opt_lang_model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss(ignore_index=4)

for i in range(epoch):
    pbar = tqdm(train_loader)
    k = 1
    j = 1
    opt_lang_model.train()
    for token_ids, tgt_ids in pbar:
        token_ids = token_ids.to(device)
        tgt_ids = tgt_ids.to(device)
        outputs = opt_lang_model(
            token_seq=token_ids,
            casual_mask=torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool().to(device),
            padding_mask=(token_ids==4).unsqueeze(-1)
        )
        loss = criterion(
            outputs.view(-1, outputs.size(-1)),
            tgt_ids.reshape(-1)
        ) / scale
        loss.backward()
        if k % scale == 0:
            k = 1
            optim.step()
            optim.zero_grad()
        else:
            k += 1
        pbar.set_description(f"Epoch {i+1}/{epoch} Loss {loss.item()*scale:.4e}")
        j += 1
        if j % 1000 == 0:
            torch.save(opt_lang_model,'./models/model'+str(i)+'.pth')
    with torch.no_grad():
        opt_lang_model.eval()
        pbar = tqdm(valid_loader)
        for token_ids, tgt_ids in pbar:
            token_ids = token_ids.to(device)
            tgt_ids = tgt_ids.to(device)
            outputs = opt_lang_model(
                token_seq=token_ids,
                casual_mask=torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool().to(device)
            )
            loss = criterion(
                outputs.view(-1, outputs.size(-1)),
                tgt_ids.reshape(-1)
            )
            pbar.set_description(f"Epoch {i+1}/{epoch} Loss {loss.item():.4e}")
    torch.save(opt_lang_model,'./models/model'+str(i)+'.pth')